In [1]:
from __future__ import print_function
print(__doc__)

import numpy as np
import time

import spacy
from grakel import GraphKernel, Graph

# Load the spaCy English model
nlp = spacy.load("en_core_web_sm")

def generate_dependency_graph(sentence):
    # Parse sentence and construct dependency tree using spaCy
    doc = nlp(sentence)
    
    # Transform dependency tree to grakel Graph representation
    edges = []
    node_labels = dict()

    for token in doc:
        if token.dep_ != 'punct':  # Exclude punctuation dependencies
            head_node = (token.head.text, token.head.i)
            dep_node = (token.text, token.i)
            
            edges.append((head_node, dep_node, {'relation': token.dep_}))
            node_labels[head_node] = token.head.text
            node_labels[dep_node] = token.text
    
    return Graph(edges, node_labels=node_labels)

def find_most_similar_sentence(target_sentence, sentences):
    # Generate dependency graph for the target sentence
    target_graph = generate_dependency_graph(target_sentence)
    
    # Generate dependency graphs for all sentences
    all_graphs = [generate_dependency_graph(sentence) for sentence in sentences]

    # Apply Weisfeiler-Lehman graph kernel
    gk = GraphKernel(kernel={"name": "weisfeiler_lehman", "n_iter": 5})
    kernel_matrix = gk.fit_transform(all_graphs)

    # Calculate similarity scores
    similarity_scores = kernel_matrix[:, sentences.index(target_sentence)].reshape(-1)

    # Exclude similarity to itself
    similarity_scores[sentences.index(target_sentence)] = 0.0

    # Find the index of the most similar sentence
    most_similar_index = np.argmax(similarity_scores)
    
    return sentences[most_similar_index]

# Example usage
sentences = ["Which magazine was started first Arthur's Magazine or First for Women?",
             "The Oberoi family is part of a hotel company that has a head office",
             "In what city? Musician and satirist Allie Goertz wrote a song about the The Simpsons character Milhouse, who Matt Groening named after who?"]

for target_sentence in sentences:
    most_similar_sentence = find_most_similar_sentence(target_sentence, sentences)
    print(f"Target Sentence: {target_sentence}")
    print(f"Most Similar Sentence: {most_similar_sentence}\n")


Automatically created module for IPython interactive environment


c:\Users\1J1870897\Anaconda3\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.6.0) was trained with spaCy v3.6.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Target Sentence: Which magazine was started first Arthur's Magazine or First for Women?
Most Similar Sentence: Which magazine was started first Arthur's Magazine or First for Women?

Target Sentence: The Oberoi family is part of a hotel company that has a head office
Most Similar Sentence: In what city? Musician and satirist Allie Goertz wrote a song about the The Simpsons character Milhouse, who Matt Groening named after who?

Target Sentence: In what city? Musician and satirist Allie Goertz wrote a song about the The Simpsons character Milhouse, who Matt Groening named after who?
Most Similar Sentence: The Oberoi family is part of a hotel company that has a head office



In [4]:
import pandas as pd
df = pd.read_csv('../test/genai_questions_809156896092057937.csv')
sentences = list(set(df['questions'].to_list()))

for target_sentence in sentences:
    most_similar_sentence = find_most_similar_sentence(target_sentence, sentences)
    print(f"Target Sentence: {target_sentence}")
    print(f"Most Similar Sentence: {most_similar_sentence}\n")


Target Sentence: Can you provide examples of local regulations, insurance requirements, and safety standards, such as ANSI/ASME B15.0-2016, that need to be taken into account when building the anomaly model?
Most Similar Sentence: Are there any constraints on computational resources, data storage, or data transfer that need to be taken into account when building the anomaly model?

Target Sentence: Is there any historical data on past failures or anomalies in the wind turbine gearbox that can be used for training the model?
Most Similar Sentence: Are there any other systems or components that can affect the performance of the wind turbine gearbox, such as the generator or the blades?

Target Sentence: What is the recommended action when noise levels in a wind turbine gearbox system deviate by more than 5 dB from the normal range?
Most Similar Sentence: What are the specific sensors present in the wind turbine system that monitor the condition of the gearbox, such as temperature, vibrat

KeyboardInterrupt: 